**Revise published Datasets with `experimental_condition_ontology_term_id`**

Driven by a revision spec in `revisions/` (format in `revisions/README.md`) that a curator writes after reading the paper. Per listed Dataset this notebook downloads the published h5ad, adds the one obs column, strips the portal-added fields, asserts nothing else moved, validates, writes a `_revised.h5ad`, then uploads into a revision of the Collection.

Everything up to the upload runs without an API key. The upload needs `cxg-api-key.txt` in one of the locations `cellxgene_mods.get_path` searches.

In [ ]:
import anndata as ad
import pandas as pd
import requests
from pathlib import Path

import cellxgene_mods as cxgm
from cellxgene_mods import CxG_API, report

**Specify the revision spec and where to keep the files**

In [ ]:
spec_path = 'revisions/CXG-950.json'
download_dir = Path('~/Downloads/CXG-950').expanduser()

spec = cxgm.load_revision_spec(spec_path)
collection_id = spec['collection_id']
spec

**List the Datasets in the published Collection and mark which the spec touches**

In [ ]:
collection = requests.get(f'{cxgm.CXG_PUBLIC_API}/collections/{collection_id}').json()
pd.DataFrame([
    {
        'dataset_id': d['dataset_id'],
        'in_spec': d['dataset_id'] in spec['datasets'],
        'title': d['title'],
        'cells': d['cell_count'],
        'schema': d['schema_version'],
    }
    for d in collection['datasets']
]).set_index('dataset_id')

**Check every term in the spec against schema 7.1.0 before touching a file**

In [ ]:
labels = cxgm.check_experimental_condition_terms(cxgm.spec_terms(spec))

**Revise each listed Dataset**

Per Dataset: download (skipped if already present), fingerprint, add the column, strip portal fields, fingerprint again and assert that only the intended changes happened, write, re-read and confirm the round trip, validate.

In [ ]:
revised_files = {}

for dataset_id, ds in spec['datasets'].items():
    report(f'=== {dataset_id} ===')
    path = cxgm.download_dataset_h5ad(collection_id, dataset_id, download_dir)
    adata = ad.read_h5ad(path)
    before = cxgm.fingerprint_adata(adata)

    crosstab = cxgm.add_experimental_condition(adata, ds['source_column'], ds['terms'])
    display(crosstab)

    adata = cxgm.revise_cxg(adata)
    cxgm.assert_only_changes(
        before, cxgm.fingerprint_adata(adata),
        added_obs=[cxgm.EC_FIELD],
        removed_obs=cxgm.OBS_PORTAL_ALL,
        removed_var=cxgm.VAR_PORTAL_REQUIRED,
        removed_uns=cxgm.UNS_PORTAL_REQUIRED,
        changed_uns=['genetic_perturbations'],
    )

    new_path = cxgm.revised_path(path)
    adata.write(new_path, compression='gzip')
    written = cxgm.fingerprint_adata(adata)
    del adata
    cxgm.assert_only_changes(written, cxgm.fingerprint_adata(ad.read_h5ad(new_path)), ignore_uns=True)

    if not cxgm.validate(str(new_path)):
        raise RuntimeError(f'{new_path} did not validate')
    revised_files[dataset_id] = new_path

revised_files

**Create the revision** (API key needed from here on)

Reuses an already open revision rather than creating a second one.

In [ ]:
CxG_API.config()

published = CxG_API.get_collection(collection_id)
revision_id = published.get('revising_in') or CxG_API.create_revision(collection_id)
revision_id

**Upload each revised file into the revision**, carrying over a fragment file if the Dataset has one

In [ ]:
for dataset_id, new_path in revised_files.items():
    current_manifest = CxG_API.get_dataset_manifest(revision_id, dataset_id)
    uri = CxG_API.upload_local_datafile(str(new_path), revision_id, dataset_id)
    manifest = cxgm.build_upload_manifest(current_manifest, uri)
    CxG_API.upload_datafiles_from_manifest(manifest, revision_id, dataset_id)

**Compare the revision to the published Collection**

Run once the portal has finished processing the uploads. Only the revised Datasets should differ, and only in fields that follow from the new column.

In [ ]:
cxgm.compare_revision(CxG_API.get_collection(revision_id))